# Large Language Models: High-Level Analysis

This notebook demonstrates a simple analysis of a document to illustrate on a high level how large language models process text. We will:
- Load the PDF document
- Extract its text
- Analyze the frequency of each letter and sign
- Generate new text

## 1. Import Required Libraries

We will use libraries for PDF reading and text processing.

In [1]:
# Import required libraries
from PyPDF2 import PdfReader
import pandas as pd
import random
import tiktoken
from IPython.display import HTML
from collections import defaultdict, Counter

## 2. Load and Read PDF Document

We will load the document and extract its text for analysis.

In [2]:
# Load and read the PDF document
pdf_path = 'Grundgesetz.pdf'
text = ''
with open(pdf_path, 'rb') as file:
    reader = PdfReader(file)
    for page in reader.pages:
        text += page.extract_text()

## 3. Extract Text from PDF

The text from the PDF is now stored in a variable. Let's preview a part of it to verify extraction.

In [3]:
# Preview the extracted text
print(text[2886:3420])


Art 3  
(1) Alle Menschen sind vor dem Gesetz gleich.
(2) Männer und Frauen sind gleichberechtigt. Der Staat fördert die tatsächliche Durchsetzung der
Gleichberechtigung von Frauen und Männern und wirkt auf die Beseitigung bestehender Nachteile hin.
(3) Niemand darf wegen seines Geschlechtes, seiner Abstammung, seiner Rasse, seiner Sprache, seiner Heimat
und Herkunft, seines Glaubens, seiner religiösen oder politischen Anschauungen benachteiligt oder bevorzugt
werden. Niemand darf wegen seiner Behinderung benachteiligt werden.



## 4. Character Occurrence

We will compute how often each character or character set occurs in the text.

In [4]:
# Compute n-gram transition counts
n = 20  # Set the length of the n-gram (e.g., 2 for bigrams, 3 for trigrams)

ngrams = Counter()
for i in range(len(text) - n + 1):
    ngram = tuple(text[i:i+n])
    ngrams[ngram] += 1

# Get top n-grams
top_ngrams = sorted(ngrams.items(), key=lambda x: x[1], reverse=True)

# Create DataFrame for top 50 n-grams
top_df = pd.DataFrame([(str(seq), count) for seq, count in top_ngrams[:50]], columns=['Sequence', 'Count'])

# Display in chunks of 20, arranged side by side
chunk_size = 20
html_parts = []
for i in range(0, len(top_df), chunk_size):
    chunk = top_df.iloc[i:i+chunk_size]
    html_parts.append(chunk.to_html(index=False))

# Arrange in a table with 2 columns
html = '<table style="border-collapse: collapse;"><tr>'
for j, part in enumerate(html_parts):
    html += f'<td style="vertical-align: top; padding: 10px;">{part}</td>'
    # if (j + 1) % 2 == 0 and j < len(html_parts) - 1:
    #     html += '</tr><tr>'
html += '</tr></table>'

print(f"Total unique n-grams: {len(ngrams)}")
display(HTML(html))

Total unique n-grams: 162624


Sequence,Count
"('Z', 'u', 's', 't', 'i', 'm', 'm', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e')",86
"('u', 's', 't', 'i', 'm', 'm', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e', 's')",86
"('s', 't', 'i', 'm', 'm', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e', 's', 'r')",82
"('t', 'i', 'm', 'm', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e', 's', 'r', 'a')",82
"('i', 'm', 'm', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e', 's', 'r', 'a', 't')",82
"('m', 'm', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e', 's', 'r', 'a', 't', 'e')",82
"('m', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e', 's', 'r', 'a', 't', 'e', 's')",82
"(' ', 'Z', 'u', 's', 't', 'i', 'm', 'm', 'u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd')",81
"('u', 'n', 'g', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd', 'e', 's', 'r', 'a', 't', 'e', 's', ' ')",60
"('E', 'i', 'n', ' ', 'S', 'e', 'r', 'v', 'i', 'c', 'e', ' ', 'd', 'e', 's', ' ', 'B', 'u', 'n', 'd')",51


## 5. Text Generation from N-Grams

We will build a text generator based on the n-gram frequencies to create new text.

In [5]:
# Build the n-gram model for generation
model = defaultdict(Counter)
for ngram, count in ngrams.items():
    prefix = ngram[:-1]
    next_char = ngram[-1]
    model[prefix][next_char] = count


def generate_text(start, length):
    current = list(start)
    for _ in range(length):
        prefix = tuple(current[-(n-1):])
        if prefix in model and model[prefix]:
            chars, counts = zip(*model[prefix].items())
            next_char = random.choices(chars, weights=counts, k=1)[0]
            current.append(next_char)
        else:
            break  # Stop if no continuation is found
    return ''.join(current)

# Example generation
start_sequence = 'Alle Menschen sind vor dem'  
generated_text = generate_text(start_sequence, 500) 
print("Generated text:\n")
print(generated_text)

Generated text:

Alle Menschen sind vor dem Gesetz gleich.
(2) Männer und Frauen sind gleichberechtigt. Der Staat fördert die tatsächliche Durchsetzung der
Gleichberechtigung von Frauen und Männern und wirkt auf die Beseitigung bestehender Nachteile hin.
(3) Niemand darf gegen sein Gewissen zum Kriegsdienst mit der Waffe verpflichtet werden. Die Dauer des Ersatzdienstes vorsehen muß, die in keinem Zusammenhang mit der deutschen Einigung
aufgeworfenen Fragen zur Änderung oder Ergänzung des Grundgesetzes und zur Übertragung von
Hoheitsrech


# 6. Kontext Fenster

- Sind wichtig für den richtigen "Ton", die richtige Thematik und einen sinnvollen Text
- Die Anzahl an potentiellen n-gramss explodiert irgendwann und das Vorkommen in den Trainingsdaten wird irgendwann zu selten
- Man möchte nicht nur aus den Trainingsdaten kopieren, sondern intra-/extrapolieren
- Deshalb: ein Neuronales Netz lernt die Wahrscheinlichkeit für das nächste Zeichen zu "schätzen"

# 7. Tokens
- N-Grams sind nicht wirklich maschinen-lesbar
- Einzelne Buchstaben zu kodieren kann schnell zu Fehlerfortpflanzung führen
- Einzelne Buchstaben führen schnell zu großen n-grams
- LLMs arbeiten mit Tokens, was man eher mit einer Kodierung von "Silben" anstatt Buchstaben vergleichen kann
- Jede "Silbe" bekommt eine Zahl zugeordnet, sodass das Modell prinzipiell lernt Zahlenfolgen vorherzusagen

In [6]:
# Get the tokenizer for the GPT-4o model
enc = tiktoken.encoding_for_model("gpt-4o")

# Encode a string into tokens
tokens = enc.encode("Art 3  \
(1) Alle Menschen sind vor dem Gesetz gleich. \
(2) Männer und Frauen sind gleichberechtigt. Der Staat fördert die tatsächliche Durchsetzung der \
Gleichberechtigung von Frauen und Männern und wirkt auf die Beseitigung bestehender Nachteile hin. \
(3) Niemand darf wegen seines Geschlechtes, seiner Abstammung, seiner Rasse, seiner Sprache, seiner Heimat \
und Herkunft, seines Glaubens, seiner religiösen oder politischen Anschauungen benachteiligt oder bevorzugt \
werden. Niemand darf wegen seiner Behinderung benachteiligt werden.")
print(tokens) 

[8372, 220, 18, 220, 350, 16, 8, 22665, 19488, 5029, 5774, 2019, 81754, 26315, 13, 350, 17, 8, 55639, 844, 36087, 5029, 26315, 16907, 152877, 13, 8296, 62887, 6731, 66454, 1076, 193375, 9617, 32002, 59278, 1227, 76717, 16907, 3487, 28493, 2812, 36087, 844, 172787, 844, 94781, 2933, 1076, 418, 1220, 278, 28493, 96409, 1644, 186616, 14508, 13, 350, 18, 8, 194728, 48469, 41383, 83911, 16341, 44367, 2822, 11, 26438, 127025, 5518, 988, 11, 26438, 460, 5788, 11, 26438, 89476, 11, 26438, 108297, 844, 141729, 11, 83911, 123418, 696, 11, 26438, 134340, 74050, 5126, 134282, 73307, 753, 4644, 2974, 678, 23405, 6591, 5126, 146180, 83, 5285, 13, 194728, 48469, 41383, 26438, 25348, 156491, 2974, 678, 23405, 6591, 5285, 13]


In [7]:
for token in tokens:
    print(enc.decode_single_token_bytes(token))

b'Art'
b' '
b'3'
b' '
b' ('
b'1'
b')'
b' Alle'
b' Menschen'
b' sind'
b' vor'
b' dem'
b' Gesetz'
b' gleich'
b'.'
b' ('
b'2'
b')'
b' M\xc3\xa4nner'
b' und'
b' Frauen'
b' sind'
b' gleich'
b'bere'
b'chtigt'
b'.'
b' Der'
b' Staat'
b' f\xc3\xb6r'
b'dert'
b' die'
b' tats\xc3\xa4ch'
b'liche'
b' Durch'
b'setzung'
b' der'
b' Gleich'
b'bere'
b'cht'
b'igung'
b' von'
b' Frauen'
b' und'
b' M\xc3\xa4nnern'
b' und'
b' wirkt'
b' auf'
b' die'
b' B'
b'ese'
b'it'
b'igung'
b' besteh'
b'ender'
b' Nachteile'
b' hin'
b'.'
b' ('
b'3'
b')'
b' Niemand'
b' darf'
b' wegen'
b' seines'
b' Gesch'
b'lech'
b'tes'
b','
b' seiner'
b' Abst'
b'amm'
b'ung'
b','
b' seiner'
b' R'
b'asse'
b','
b' seiner'
b' Sprache'
b','
b' seiner'
b' Heimat'
b' und'
b' Herkunft'
b','
b' seines'
b' Glaub'
b'ens'
b','
b' seiner'
b' religi'
b'\xc3\xb6sen'
b' oder'
b' politischen'
b' Ansch'
b'au'
b'ungen'
b' ben'
b'ach'
b'teil'
b'igt'
b' oder'
b' bevorzug'
b't'
b' werden'
b'.'
b' Niemand'
b' darf'
b' wegen'
b' seiner'
b' Beh'
b'inderung'
b' ben'
